# ECON2041 week 6 lecture: pokies, unemployment, and gambling losses

Everything we'll run on screen today is in here, so you can follow along now and rerun the
whole hour afterward.

One row per Victorian local government area (LGA), 79 of them, for 2024.

## Data sources

- Gaming expenditure: Victorian Gambling and Casino Control Commission, 2024
- Employment: Department of Employment, Small Area Labour Markets, 2024
- Population: Department of Environment, Land, Water and Planning, 2023

## Setup

Our usual setup block, with the same `ols` import you'll use in this week's tutorial.

In [ ]:
# Our standard ECON2041 setup block
import numpy as np               # numerical tools (nicknamed np)
import pandas as pd              # data tools (nicknamed pd)
import matplotlib.pyplot as plt  # plotting tools (nicknamed plt)
import seaborn as sns            # statistical charts (nicknamed sns)
from statsmodels.formula.api import ols  # ordinary least squares regression

# Keep scalar output plain under NumPy 2 (0.5, not np.float64(0.5)).
if np.lib.NumpyVersion(np.__version__) >= "2.0.0":
    np.set_printoptions(legacy="1.25")

DATA = "https://emiliatjernstrom.com/econ2041/data"   # Unit datasets live at this web address

print("Setup done!")

## What are we asking, and what do we have?

Pokies sit in pubs and clubs across Victoria, and they are not spread evenly. Among the LGAs
that have any, pokie density (machines per 1,000 adults) ranges from just over one to just
over ten, and nine LGAs have none at all.

> Do adults lose more where there are more pokies, and does unemployment still matter once we hold pokie density fixed?

We'll use these variables from the dataset:

| Variable | What it stores |
|---|---|
| `total_net_exp` | Total net gaming expenditure |
| `adult_pop` | LGA adult population |
| `egm_per_1000` | Electronic gaming machines (EGMs, or pokies) per 1,000 adults, which we'll call pokie density |
| `exp_per_adult` | Expenditure per adult (\$) |
| `unemployed` | Unemployed persons (count) |
| `ue_rate` | LGA unemployment rate |
| `SEIFA_dis_score` | Socio-Economic Indexes for Areas (SEIFA) disadvantage score, published by the Australian Bureau of Statistics (higher = less disadvantaged) |

Net expenditure is the industry's phrase for what players lose, so the two expenditure
variables are gambling losses.

In [ ]:
# Load pokies-victoria-2024.csv into a dataframe called pokies
pokies = pd.read_csv(f"{DATA}/pokies-victoria-2024.csv")

pokies.head()

## Get to know the data

In [ ]:
# How many LGAs, and how many variables?
pokies.shape

In [ ]:
# Summary numbers for the four variables we'll work with today
pokies[["exp_per_adult", "egm_per_1000", "ue_rate", "SEIFA_dis_score"]].describe().round(2)

### Where do adults lose the most?

In [ ]:
# Sort from most lost per adult to least, and show the top 10
pokies.sort_values(by="exp_per_adult", ascending=False).head(10)[
    ["LGA", "exp_per_adult", "egm_per_1000", "ue_rate"]
]

### And the other end?

In [ ]:
# Same sort, other end. 12 rows: all nine zero-pokie rural shires, plus a few nonzero neighbors
pokies.sort_values(by="exp_per_adult", ascending=True).head(12)[
    ["LGA", "exp_per_adult", "egm_per_1000", "ue_rate"]
]

## How much do losses vary across LGAs?

In [ ]:
# A histogram of losses per adult, with the mean and the median marked
mean_loss = pokies["exp_per_adult"].mean()
median_loss = pokies["exp_per_adult"].median()

sns.histplot(data=pokies, x="exp_per_adult", bins=11, edgecolor="black", linewidth=0.7)
plt.axvline(mean_loss, color="hotpink", linewidth=2)          # the mean
plt.axvline(median_loss, color="black", linestyle="--")       # the median
plt.xlabel("Gambling losses per adult ($/year)")
plt.ylabel("Number of LGAs")
plt.show()

print(f"mean: {round(mean_loss)}, median: {round(median_loss)}")

## Which variable correlates most strongly with losses?

We predicted in the portal which variable would correlate most strongly with losses. Now
we'll check.

Same `corr()` you used in the week 5 tutorial, asked of four columns at once.

In [ ]:
# Every pairwise correlation among the four variables
pokies[["exp_per_adult", "egm_per_1000", "ue_rate", "SEIFA_dis_score"]].corr().round(2)

Read down the `exp_per_adult` column. The top entry is 1.00, because every variable
correlates perfectly with itself. The three entries below it give each other variable's
correlation with losses per adult.

## See the relationship

Every dot is one LGA.

In [ ]:
# Pokie density on the horizontal axis, losses per adult on the vertical
sns.scatterplot(data=pokies, x="egm_per_1000", y="exp_per_adult", s=40, alpha=0.7)
plt.xlabel("EGMs per 1,000 adults")
plt.ylabel("Gambling losses per adult ($/year)")
plt.show()

## Fit a regression

Same two lines you'll use in the week 6 tutorial: the outcome goes to the left of `~`, the
explanatory variable to the right.

In [ ]:
# Fit losses per adult on pokie density with ordinary least squares
model = ols("exp_per_adult ~ egm_per_1000", data=pokies).fit()

model.params.round(1)

### The whole output table

`.params` gives the two numbers we want. `.summary()` gives everything statsmodels computed,
including the columns we'll come back to in week 9.

In [ ]:
print(model.summary())

### The fitted line, drawn through all 79 LGAs

`sns.regplot` draws the scatterplot and the least-squares line in one command.

In [ ]:
sns.regplot(data=pokies, x="egm_per_1000", y="exp_per_adult", ci=None,
            scatter_kws={"s": 40, "alpha": 0.7},
            line_kws={"color": "orange"})
plt.xlabel("EGMs per 1,000 adults")
plt.ylabel("Gambling losses per adult ($/year)")
plt.show()

## Does unemployment still matter, once we hold pokie density fixed?

Split the LGAs at the median unemployment rate and look at the two halves separately.
`ue_rate` stores the unemployment rate as a proportion, so 0.05 means 5%.

In [ ]:
# The median unemployment rate
median_val = pokies["ue_rate"].median()

print(f"median unemployment rate: {round(median_val, 4)}")

### Building a dummy variable

`>` asks a yes-or-no question of every LGA at once, and `.astype(int)` turns the answers
into 1 and 0. A 0/1 variable like this is called a dummy variable.

In [ ]:
# high_ue stores the comparison's answer, as 1 (above median) or 0 (below)
pokies["high_ue"] = (pokies["ue_rate"] > median_val).astype(int)

# A readable label for plotting, and one color per group so every figure matches
pokies["ue_group"] = np.where(pokies["high_ue"] == 1, "High unemployment", "Low unemployment")
UE_ORDER = ["Low unemployment", "High unemployment"]
UE_COLORS = {"Low unemployment": "#5b8fc9", "High unemployment": "#c8102e"}

pokies[["LGA", "ue_rate", "high_ue", "ue_group"]].head()

In [ ]:
# How many LGAs are in each group?
pokies["ue_group"].value_counts()

### One regression per group

Same model, fit twice, on the two halves of the data.

In [ ]:
# Two datasets, one per group
low_group = pokies[pokies["high_ue"] == 0]
high_group = pokies[pokies["high_ue"] == 1]

model_low = ols("exp_per_adult ~ egm_per_1000", data=low_group).fit()
model_high = ols("exp_per_adult ~ egm_per_1000", data=high_group).fit()

print("low unemployment, n =", int(model_low.nobs))
print(model_low.params.round(1))
print()
print("high unemployment, n =", int(model_high.nobs))
print(model_high.params.round(1))

### The same 79 LGAs, with a line each

`sns.lmplot` is `regplot` plus a `hue` argument, which tells seaborn to split the data by a
column and give each group its own color: one scatterplot and one fitted line per group.

In [ ]:
sns.lmplot(data=pokies, x="egm_per_1000", y="exp_per_adult", hue="ue_group",
           hue_order=UE_ORDER, palette=UE_COLORS, ci=None,
           scatter_kws={"s": 40, "alpha": 0.7})
plt.xlabel("EGMs per 1,000 adults")
plt.ylabel("Gambling losses per adult ($/year)")
plt.show()

## One regression, both groups

Splitting the sample throws away half the data for each line. Putting the dummy into the
model keeps all 79 LGAs and estimates one number for the difference between the groups, but
it forces one common slope on both.

In [ ]:
# A second explanatory variable, separated by a +
multi = ols("exp_per_adult ~ egm_per_1000 + high_ue", data=pokies).fit()

multi.params.round(1)

In [ ]:
print(multi.summary())

### Two parallel lines

The estimate on `high_ue` is the vertical distance between the two lines: same slope, two
intercepts.

We'll build each line from the estimates by hand, the same way you'll predict with
`model.params` in the week 6 tutorial.

In [ ]:
# Pull the three estimates out by name
b0 = multi.params["Intercept"]
b1 = multi.params["egm_per_1000"]
b2 = multi.params["high_ue"]

# 100 evenly spaced pokie-density values to draw the lines over
egm_grid = np.linspace(0, pokies["egm_per_1000"].max(), 100)

sns.scatterplot(data=pokies, x="egm_per_1000", y="exp_per_adult", hue="ue_group",
                hue_order=UE_ORDER, palette=UE_COLORS, s=40, alpha=0.7)
plt.plot(egm_grid, b0 + b1 * egm_grid, color=UE_COLORS["Low unemployment"])          # high_ue = 0
plt.plot(egm_grid, b0 + b2 + b1 * egm_grid, color=UE_COLORS["High unemployment"])    # high_ue = 1
plt.xlabel("EGMs per 1,000 adults")
plt.ylabel("Gambling losses per adult ($/year)")
plt.show()

print(f"vertical distance between the lines: {round(b2, 1)}")

## A second explanatory variable need not be a dummy

Multiplying `ue_rate` by 100 gives us a coefficient we can read per percentage point of
unemployment.

In [ ]:
# ue_pct stores the unemployment rate in percentage points
pokies["ue_pct"] = 100 * pokies["ue_rate"]

multi_ue = ols("exp_per_adult ~ egm_per_1000 + ue_pct", data=pokies).fit()

multi_ue.params.round(1)

### Three models side by side

`pd.DataFrame` unions the three models' parameter names, so a blank cell means that variable
is not in that specification, not a missing value. Read across the `egm_per_1000` row: the
estimate is 78.3 in the simple model, 75.9 once the dummy is added, and 72.8 once unemployment
enters as a percentage, a difference that week 7 explains as omitted-variable bias.

In [ ]:
# Collect the estimates from all three models into one table
comparison = pd.DataFrame({
    "simple": model.params,
    "with dummy": multi.params,
    "with rate": multi_ue.params,
}).round(1)

comparison

## Your turn

Three things worth trying before the week 7 quiz:

1. Swap `SEIFA_dis_score` in for `ue_pct` and compare the coefficient on `egm_per_1000` across the two specifications
2. Split at the mean unemployment rate instead of the median, and compare the dummy's estimate across the two splits
3. Drop the nine LGAs with no pokies, fit the simple regression again, and compare the slope with and without them

Each of these will give you a different estimate, and nothing in the data picks the right
specification for you. Pokies are not randomly assigned across LGAs, so none of today's
numbers are causal. You have to be able to defend the choice you make.